# Case study: Tides, storm surge and sea level

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import matplotlib

In [ ]:
# Fill in this cell with your personal details:
# - Name: xxx
# - Student ID: xxx
# - Email: xxx

**Deadline.** December 18, 2026. Submit your completed notebook on Gradescope.

**Rules.**
- Work alone. You may not discuss the case study or collaborate on it with other students.
- AI assistants are allowed without restriction. You must be able to explain everything you submit: you lose points for any line of code or any choice that you cannot justify.
- All cells must run from top to bottom, and the submitted notebook must keep their outputs.

**Evaluation.**
- During the January exam session, you explain and defend your solution in a 30-minute oral discussion, separate from the oral exam on theory. You may be asked about any part of your code and choices, and to modify or extend your analysis.
- The case study accounts for 40% of the final grade, and the oral exam on theory for the other 60%. A minimum of 8/20 is required for each. Below that, the lowest of the two grades becomes the final grade.

**How to answer.**
- Follow the Bayesian workflow of the course (build, compute, critique, repeat) and its principles of data visualization.
- Most questions are split into points A, B, C, and so on. Start your answer to each point with a markdown cell that holds only its title, for example `### QUESTION 2.A`. For a question without points, the title is `### QUESTION 5`. Add as many cells as you need.
- Explain your reasoning, choices and interpretations in markdown cells and code comments.
- Use the Python scientific ecosystem (`numpy`, `scipy`, `pandas`, `matplotlib`), and `jax` for the filter of Q6 and the estimation of Q8.
- Some cells are expensive. Write your code so that you can re-run a cell without re-running everything before it.

# Introduction

This case study is about one year of sea level in the harbour of Boston, on the east coast of the United States, measured by a tide gauge a few times per day, at irregular times. The sea there rises and falls by about three metres twice a day, and a few times a year a storm pushes it high enough to flood the streets near the harbour. The measured level is the sum of several effects that cannot be observed separately. Your job is to separate them, with a model that also says how sure it is about each one.

## What moves the sea

**1. The tide.** The Moon pulls on the Earth and its oceans, more strongly on the side facing it than on the far side. The ocean bulges in two places, one facing the Moon and one on the opposite side. As the Earth turns, a harbour passes through both bulges and gets two high waters. The Moon also moves along its orbit, by about 13 degrees per day, so the Earth needs about 50 more minutes to bring the harbour back under it. The same harbour faces the Moon every 24 h 50 min, a **lunar day**, and its high waters come every 12 h 25 min. This main part of the tide is called **M2**. Your measurements are too far apart to show it directly, but the periodogram of Question 1 finds it. A periodogram rebuilds the record as a sum of sine waves and tells how big each one must be.

The Sun does the same, with about half the strength of the Moon. It comes back over the same place every 24 h, so its tide, called **S2**, has a period of exactly 12 h. The two tides go in and out of step every 14.77 days. In step, they add up and the tide is large; out of step, they partly cancel and the tide is small. The animation below shows this.

<img src="tides.gif" width="800">

Two more effects matter. The distance between the Earth and the Moon changes over 27.55 days, so the tide is larger when the Moon is closest and smaller when it is farthest. The orbit of the Moon is also tilted: the Moon is usually north or south of the equator, by up to 28 degrees. The two bulges are then centred north and south of the equator, and the two high waters of a day are unequal. This inequality vanishes every two weeks, when the Moon crosses the equator.

<img src="tilt_explained.png" width="900">

A wave with two cycles per day always gives two equal high waters. To make them unequal, the tide also needs a wave with one cycle per day, the dotted curve in the figure.

Each of these effects adds one sine wave with its own period, called a **constituent**. M2 is the largest almost everywhere. In Boston, the second largest comes from the distance of the Moon. The periods of the constituents follow from astronomy and are known precisely before seeing any data; in Question 3, you compute about ten of them from four astronomical numbers. Their size and timing depend on the shape of the coast and the depth of the sea: the same Moon gives a tide of 0.3 m in the Mediterranean, 3 m in Boston and 16 m in the Bay of Fundy. Size and timing must therefore be learned from the data.

The harbour itself adds constituents. In shallow water, the tide is no longer a pure sine wave, but it still repeats every 12 h 25 min. A signal that repeats every $T$ hours is a sum of sine waves with periods $T$, $T/2$, $T/3$, and so on (a Fourier series). The first of these extra waves, at 6 h 13 min, is called **M4**. It is in the table of Question 3, although in Boston it is only a few centimetres.

**2. The storm surge.** The weather moves the sea too. When the air pressure drops by 1 hPa, the sea rises by about 1 cm, so a storm at 980 hPa lifts the sea by about 33 cm above its level at a normal 1013 hPa. You will check this number on the data. Wind blowing towards the coast also pushes water against it. This rise of the sea caused by the weather is called the **surge**. It builds up over a few hours when a storm arrives, and goes back to zero over a day or two once the storm has passed. It does not repeat, so no calendar can predict it, and it is what floods harbours.

**3. The yearly cycle.** Water expands when it warms, and winds and sea currents change with the seasons. In Boston, the average sea level goes up and down over the year, by about 10 cm from its lowest to its highest. This cycle repeats with a known period, one year, so it is handled like a tidal constituent.

**4. The fast movements.** Waves, and the water sloshing back and forth inside the harbour, move the sea within an hour. With measurements several hours apart, these movements look like random noise from one measurement to the next. The error of the instrument adds a little to this noise. The noise term of the model collects both.

## Why this is a data science problem

The gauge measures only the sum of these four effects, and no instrument measures the tide or the surge separately. Yet the split matters. Tide tables can be printed years in advance, because the tide repeats. Flood warnings need the surge, which does not.

The tide, the surge and the yearly level are latent states that change over time, and the measurement is their sum plus noise. This is the setting of the state-space models of Lecture 5. The classical method, in use for more than a century, fits sine waves of fixed size at the known periods by least squares, and calls what is left the surge. It gives no uncertainty on the surge and cannot forecast it. A state-space model does both.

You will explore the data, build a model from the effects above, compute the posterior of the hidden components with the Kalman filter and smoother, estimate the parameters, separate the tide, the surge and the mean level, and finally check the model and forecast the water level.

# Data exploration

You are given the file `water-level-irregular.csv`. It contains one year of measurements from the tide gauge of Boston, station 8443970 of the American agency NOAA, from 1 January to 31 December 2024. The measurements are not equally spaced. They were picked at random from the hourly record of the gauge, with 3 to 9 hours between two measurements, so every time is a whole hour. Times are in UTC, so there is no change of hour for summer time.

**`x`** is the measured sea level, in metres, at the time of the measurement. It is a height above a fixed reference level, called the zero level of the station. NOAA computed this level once, as the average of all measurements made at this station between 1983 and 2001. Averaging over nineteen years removes the slow cycles of the Moon, the longest of which lasts 18.6 years. Sea level has risen since that period, so the average of `x` over 2024 is not zero. The size of this shift is not given to you: your model has to estimate it.

**`x_pred`** is the water level that NOAA predicts at the time of each measurement from the tide and the yearly cycle alone, in metres, computed from many years of past measurements at this station. It does not contain the surge, nor the rise of the mean sea level since 1983–2001. **You are not allowed to use it to fit your model.** It is kept aside so that, at the end, you can compare the tide found by your model with the tide found by a different method. You may use it in Question 1 to remove the tide and look at what is left, as long as you say so.

**`p`** is the air pressure, in hPa, measured by a sensor on the gauge itself, at the same place as the sea level. It is missing for a few measurements.

<div class="alert alert-success">

**Q1**. Load the data from `water-level-irregular.csv` into a Pandas DataFrame, and explore it. At least:

**A.** Check every column for missing values, and say where they are. Look also at the time between two consecutive measurements.

**B.** Plot the water level over four timescales: the whole year, two months, one week and two days, with one dot per measurement. Comment on what each timescale shows, and on what it no longer shows now that the measurements are several hours apart.

**C.** Plot the periodogram of the record. The measurements are not equally spaced, so use a Lomb-Scargle periodogram, which fits a sine wave by least squares at each frequency. `scipy.signal.lombscargle` takes angular frequencies; with `normalize='amplitude'`, it returns complex numbers whose modulus is the amplitude of the sine wave. Identify the lines that stand out, and relate them to the lunar day (24.84 h) and the solar day (24 h). The irregular sampling spreads part of each line over other frequencies, as smaller copies around it (side lobes) and false peaks elsewhere, of 10 to 15 cm, larger than some real constituents. To tell them apart, compute the periodogram of a pure M2 sine wave sampled at the same times, and compare. State the frequency resolution of your periodogram, the smallest difference between two frequencies that the record can separate, about one over its length. What does it imply about the lines you can and cannot separate?

**D.** Your periodogram shows lines near one cycle per day, although the sea rises twice a day. Explain where they come from, with the figure of the introduction.

**E.** Plot `x - x_pred` over the whole year. With the predicted tide removed, what is left is mostly the weather. When is it largest? Use the pressure to check that the largest peaks are storms. Find the measurement with the largest value of `x - x_pred`, and the measurement with the highest water level `x`. Are they the same?

</div>

# Build: a state-space model of the water level

## Continuous time

We first describe the system in continuous time, and discretize it afterwards, as recommended in [Lecture 5](https://glouppe.github.io/dats0001-foundations-of-data-science/?p=lecture5.md#continuous-or-discrete). Time $t$ is measured in hours.

The **tide** is a sum of sine waves, one per constituent, as in the introduction. With $J$ constituents,
$$\text{tide}(t) = \sum_{j=1}^{J} \alpha_j \cos(\omega_j t - \phi_j),$$
where constituent $j$ has a known angular frequency $\omega_j$, an unknown amplitude $\alpha_j$ (its size) and an unknown phase $\phi_j$ (its timing). The angular frequency is $\omega_j = 2\pi / (\text{period in hours})$, in radians per hour; for M2, $\omega_j = 2\pi / 12.42$. You compute all of them in Question 3, where you also choose which constituents to include, and therefore $J$.

In a state-space model of Lecture 5, each component is a state that moves from one step to the next by a linear rule, plus a little noise. A cosine is not of that form. It becomes so if we see it as a coordinate of a point turning around a circle.

Consider a point that turns around a circle of radius $\alpha_j$ at constant speed, one full turn per period of the constituent. At time $t$, its angle is $\omega_j t - \phi_j$. Its horizontal coordinate is
$$c_j(t) = \alpha_j \cos(\omega_j t - \phi_j),$$
which is the contribution of constituent $j$ to the water level. Its vertical coordinate is
$$e_j(t) = \alpha_j \sin(\omega_j t - \phi_j).$$

<img src="rotating_point.png" width="900">

The water level uses only $c_j$. We still need $e_j$, because $c_j$ alone does not say where the point goes next: the same horizontal position is reached once while $c_j$ goes up and once while it goes down, and only $e_j$ tells which.

The pair $r_j(t) = (c_j(t), e_j(t))$ is the position of the point. It contains the size and the timing of the constituent: the size $\alpha_j$ is the distance of the point to the centre, and the timing $\phi_j$ fixes where the point is on the circle at a given time. So $\alpha_j$ and $\phi_j$ do not need to be parameters of the model. We put $r_j$ in the state, the Kalman filter estimates the position of the point, and the size and the timing come with it.

The derivatives of $c_j$ and $e_j$ with respect to time are $\frac{dc_j}{dt} = -\omega_j e_j$ and $\frac{de_j}{dt} = \omega_j c_j$, a linear rule. With small random pushes added, the turning point follows the stochastic differential equation
$$dr_j(t) = \Omega_j \, r_j(t) \, dt + \sigma_\text{tide} \, dW(t), \qquad \Omega_j = \begin{bmatrix} 0 & -\omega_j \\ \omega_j & 0 \end{bmatrix}.$$
where $W(t)$ is a Brownian motion, as in Lecture 5: over a short time $dt$, the push $dW(t)$ is a Gaussian draw of variance $dt$, independent of the past. The first term turns the point at speed $\omega_j$. The second term pushes it at random, with $\sigma_\text{tide}$ in m/$\sqrt{\text{h}}$. The pushes slowly change its distance to the centre and its position on the circle, so the size and the timing of the constituent can drift during the year. With $\sigma_\text{tide} = 0$, the point turns forever on the same circle: this is a sine wave of fixed size and timing, as in the classical method of the introduction.

The **surge** $s(t)$ is pushed away from zero when a storm arrives, and relaxes back once the storm has passed. We describe it with the Ornstein-Uhlenbeck process that described the wolf in [Lecture 5](https://glouppe.github.io/dats0001-foundations-of-data-science/?p=lecture5.md#ou-process), in one dimension and with equilibrium point $0$,
$$ds(t) = -\kappa \, s(t) \, dt + \sigma_s \, dW(t),$$
where $1/\kappa$ [h] is the relaxation time of the surge, the time it needs to shrink by a factor $e$ once the weather is calm, and $\sigma_s^2 / (2\kappa)$ is its variance in the long run, with $\sigma_s$ in m/$\sqrt{\text{h}}$. The effect of the air pressure is written separately, as the term $\beta \tilde{p}(t)$ of the observation below, where $\tilde{p}$ is the air pressure minus its average. So $s(t)$ is the part of the surge that the local air pressure does not explain, such as the push of the wind, and the total surge is $s(t) + \beta \tilde{p}(t)$.

The **yearly cycle** repeats every year, so we write it like the tide, as sine waves with known periods. A single sine wave with a period of one year gives a symmetric shape, six months up and six months down. A second sine wave, with a period of half a year, lets the cycle rise faster than it falls, or stay high longer than it stays low. The two sine waves are two more constituents of the sum above, called Sa and Ssa in Question 3. Their turning points turn like those of the other constituents, with $\omega_j = 2\pi / (1 \text{ year})$ and $\omega_j = 2\pi / (\text{half a year})$, but they receive no random pushes. They turn on a fixed circle, so the yearly cycle keeps the same size and timing all year.

The **mean level** is a constant $\ell$, the shift between the zero level of the station and the average level of the sea in 2024. It does not change, so $d\ell(t) = 0$.

Each $W(t)$ above is a Brownian motion of its own, independent of all the others, and the two coordinates of each $r_j$ receive independent pushes.

## Discrete time

From now on, $t$ counts the measurements, and $\Delta t$ is the time between measurement $t-1$ and measurement $t$, between 3 and 9 h. Each equation above can be solved exactly over one step, so we can write how each component moves from one measurement to the next without approximation. Since $\Delta t$ changes from one step to the next, so do the transitions, as for the wolf in [Lecture 5](https://glouppe.github.io/dats0001-foundations-of-data-science/?p=lecture5.md#ou-discretization). For the surge, the solution is the exact discretization of the Ornstein-Uhlenbeck process of Lecture 5, with $\mu = 0$. For the tide, the solution is a rotation. You do not need to derive it.

- **Tide.** In one step, each turning point turns by the angle $\omega_j \Delta t$, and the random pushes add a small noise,
$$r_{j,t} = \text{Rot}(\omega_j \Delta t)\, r_{j,t-1} + w_{j,t}, \qquad w_{j,t} \sim \mathcal{N}\left(0, \sigma_\text{tide}^2 \Delta t \, I_2\right), \qquad \text{Rot}(a) = \begin{bmatrix} \cos a & -\sin a \\ \sin a & \cos a \end{bmatrix}.$$
For Sa and Ssa, which receive no pushes, $w_{j,t} = 0$.
- **Surge.** In one step, the surge keeps a fraction $e^{-\kappa \Delta t}$ of its value and receives a new random push,
$$s_t = e^{-\kappa \Delta t}\, s_{t-1} + \eta_t, \qquad \eta_t \sim \mathcal{N}\left(0, \frac{\sigma_s^2}{2\kappa}\left(1 - e^{-2\kappa \Delta t}\right)\right).$$
- **Mean level.** It does not move, $\ell_t = \ell_{t-1}$.

We stack all the components into the latent state
$$z_t = (c_{1,t}, e_{1,t}, \ldots, c_{J,t}, e_{J,t}, s_t, \ell_t) \in \mathbb{R}^{2J + 2}.$$
The three transitions above then form a single linear Gaussian transition, as in [Lecture 5](https://glouppe.github.io/dats0001-foundations-of-data-science/?p=lecture5.md#lgssm) with $b = 0$,
$$p(z_t | z_{t-1}) = \mathcal{N}(z_t | A z_{t-1}, Q),$$
where $A$ and $Q$ are computed with the $\Delta t$ of that step. $A$ contains the rotations, the factor $e^{-\kappa \Delta t}$ and a $1$ for the mean level; $Q$ contains the noise variances.

The measured water level $x_t$ is the sum of the components, plus the effect of the air pressure and a noise term that collects the fast movements and the error of the instrument,
$$x_t \mid z_t \sim \mathcal{N}(H z_t + \beta \tilde{p}_t, R), \qquad R = \sigma_x^2.$$
Since $x_t$ is a single number, $H$ is a single row, with one entry per component of $z_t$. It has a $1$ in front of each $c_{j,t}$, of $s_t$ and of $\ell_t$, and a $0$ in front of each $e_{j,t}$, so that
$$H z_t = \sum_{j=1}^{J} c_{j,t} + s_t + \ell_t.$$
The covariate $\tilde{p}_t = p_t - \bar{p}$ [hPa] is the measured air pressure minus its average over the year, and $\beta$ [m/hPa] is an unknown parameter.

The state holds what is hidden and needed, either to build the water level now or to move it to the next measurement. Each $c_{j,t}$ is part of the water level. Each $e_{j,t}$ is not, but without it $c_{j,t}$ cannot be moved forward, since we would not know whether it goes up or down. The surge $s_t$ has memory: a surge at one moment is still partly there a few hours later. The mean level $\ell_t$ is unknown, and putting it in the state lets the Kalman filter estimate it. The air pressure is not in the state, because it is measured. Neither is the observation noise, because it has no memory from one measurement to the next.

The parameters of the model are
$$\theta = (\kappa, \sigma_s, \sigma_\text{tide}, \sigma_x, \beta).$$

<div class="alert alert-success">

**Q2**. Build the model in code.

Write a function `build_A_Q(theta, dt, omega)` that returns $A$ and $Q$ for a time step `dt`, with the exact transitions of the Build, as [Lecture 5](https://glouppe.github.io/dats0001-foundations-of-data-science/?p=lecture5.md#ou-discretization) and `nb05` do for the wolf, and a function `build_H(J)` that returns $H$. Here `omega` holds the angular frequencies $\omega_j$ of the $J$ constituents, and both functions follow the order of $z_t$. `build_A_Q` must also handle Sa and Ssa, whose blocks of $Q$ are zero; you can recognize them by their $\omega_j$. You compute the frequencies precisely in Q3. For now, test your functions with M2 and S2 only, with the periods of the introduction (12 h 25 min and 12 h) and any values of $\theta$, for example $\kappa = 0.05$, $\sigma_s = 0.03$, $\sigma_\text{tide} = 0.002$, $\sigma_x = 0.03$ and $\beta = -0.01$. Print $A$, $Q$ and $H$, and explain why $A$ is block diagonal. Write both functions with `jax.numpy`, not `numpy`: the filter of Q6 calls `build_A_Q` inside `jax.lax.scan`, and Q8 takes its gradient with respect to $\theta$. JAX arrays cannot be modified in place, so build $A$ and $Q$ with `A.at[i, j].set(v)`, or assemble the blocks with `jax.scipy.linalg.block_diag`.

*Tip.* Write $A$ and $Q$ for $J = 1$ on paper first, with $z_t = (c_t, e_t, s_t, \ell_t)$. Each row of $A$ says how one component of the new state $z_t$ is made from the components of the old state $z_{t-1}$, and $Q$ holds the variances of the noises.

</div>

<div class="alert alert-success">

**Q3**. Compute the frequencies of the constituents, and choose them.

The frequencies of the constituents are not estimated from the data. They come from the motions of the Earth, the Moon and the Sun, which are known very precisely; each constituent combines one or two of these motions. Work with frequencies $f$ in cycles per day, $f = 1/\text{period}$.

| motion | period |
| --- | --- |
| the Earth turns once with respect to the Sun | 24 h |
| the Earth goes once around the Sun | 365.2422 days |
| the Moon goes once around the Earth | 27.321582 days |
| the distance between the Earth and the Moon goes through one cycle | 27.554550 days |

| constituent | frequency |
| --- | --- |
| S2 | $2 f_\text{day}$ |
| K1 | $f_\text{day} + f_\text{year}$ |
| P1 | $f_\text{day} - f_\text{year}$ |
| K2 | $2 f_\text{K1}$ |
| M2 | $2 (f_\text{K1} - f_\text{Moon})$ |
| N2 | $f_\text{M2} - f_\text{distance}$ |
| O1 | $f_\text{M2} - f_\text{K1}$ |
| M4 | $2 f_\text{M2}$ |
| Sa | $f_\text{year}$ |
| Ssa | $2 f_\text{year}$ |

K1 makes one cycle per turn of the Earth with respect to the stars, which takes 23 h 56 min. This is a little shorter than a day, because the Earth also moves around the Sun. M2 makes two cycles per lunar day, the 24 h 50 min of the introduction.

The angular frequencies $\omega_j$ of the Build, the `omega` of `build_A_Q`, follow as $\omega_j = 2\pi f_j / 24$ in rad/h.

**A.** Compute, for the ten constituents, the frequency in cycles per day, the period in hours and $\omega_j$ in rad/h, and show them in a table. Check that M2 and S2 give the periods of the introduction.

**B.** Draw the ten frequencies as vertical lines on your periodogram of Q1. Which ones fall on a peak, and why do the others not show?

**C.** Two constituents can be separated only by a record longer than $1/|f_1 - f_2|$, by the frequency resolution of Q1.C. Compute this length for the pairs M2/S2, M2/N2, S2/K2, K1/P1 and Sa/Ssa, and compare it with the length of your record. The record holds just one cycle of Sa. Can the model tell the mean level $\ell$ apart from Sa? Can it tell the yearly cycle apart from a slow change of the weather during 2024? Check your first answer after Q8, with the posterior correlation between $\ell$ and the states of Sa. Decide which constituents you keep. A constituent you drop does not disappear from the data: say what happens to it. You may revisit this choice after Q8, once you can compare fits.

</div>

<div class="alert alert-success">

**Q4**. Define a prior $p(z_1) = \mathcal{N}(z_1 | m_1^-, P_1^-)$ over the initial state and a prior over $\sigma_x$, and simulate from the model. The filter of Q6 starts from $p(z_1)$, before it has seen any data.

**A.** $P_1^-$ has to be wide enough to contain any plausible size and timing of the constituents. On $\ell$, it has to contain the shift between the zero level of the station and the mean sea level of 2024, which can reach a few tens of centimetres. Choose $m_1^-$ and $P_1^-$, and justify each value. The gauge sees the sum of the constituents, so judge your prior on that sum too.

**B.** Why is it safer to choose $P_1^-$ too wide than too narrow?

**C.** With measurements several hours apart, the observation noise is hard to tell apart from the surge and from the drift of the tide. A prior on $\sigma_x$ helps to estimate it in Q8. The gauge itself is far more accurate than any $\sigma_x$ you will estimate. Say what $\sigma_x$ absorbs in a record of measurements several hours apart, and choose its prior accordingly, as a distribution over $\log \sigma_x$, the scale on which you fit it.

**D.** Write functions to sample from $p(z_t|z_{t-1})$ for a given $\Delta t$ and from $p(x_t|z_t)$, and a function to simulate a full trajectory $z_{1:T}, x_{1:T}$ at the measurement times of the data, given $\theta$ and an initial distribution $p(z_1)$. The observation needs the air pressure at every step, and the pressure has a few missing values, found in Q1. Decide how to fill them, and say why. Then simulate a few records with values of $\theta$ of your choice, for example those of Q2, and $z_1$ drawn from your prior, and compare them with the data. Is your prior plausible?

</div>

### QUESTION 4.A

### QUESTION 4.B

### QUESTION 4.C

### QUESTION 4.D

In [ ]:
def simulate(key, theta, omega, H, m1, P1, dts, ps):
    """Simulates one record at the given time steps.

    key = a jax.random key
    dts = time since the previous step [h], 0 at the first step
    ps = air pressure minus its average [hPa]
    Returns z (the states, one row per step) and x (the water levels)."""
    ...

<div class="alert alert-success">

**Q5**. Draw the probabilistic graphical model. What are the observed variables? What are the latent variables? What are the parameters? Which quantities are fixed and known? Write the factorization of the joint distribution, and count how many scalar latent variables the model has at the measurement times. You may draw the model by hand and include a photo of the drawing.

</div>

# Compute

The model is a linear Gaussian state-space model, so the Bayes filter and the Bayes smoother of Lecture 5 have closed forms: the [Kalman filter](https://glouppe.github.io/dats0001-foundations-of-data-science/?p=lecture5.md#kalman-filter) and the [Rauch-Tung-Striebel smoother](https://glouppe.github.io/dats0001-foundations-of-data-science/?p=lecture5.md#kalman-smoother).

<div class="alert alert-success">

**Q6**. Implement the Kalman filter and the Rauch-Tung-Striebel smoother.

**A.** Implement the Kalman filter. It returns the filtering distributions $p(z_t|x_{1:t}) = \mathcal{N}(z_t|m_t, P_t)$, the one-step-ahead predictions $\mathcal{N}(z_t|m_t^-, P_t^-)$, and the innovations $\nu_t = x_t - Hm_t^- - \beta\tilde{p}_t$, the errors of the one-step-ahead predictions of $x_t$, with their variances $S_t = HP_t^-H^T + R$. Use `jax` and `jax.lax.scan`: the state has dimension $2J + 2$, the record has 1443 steps, and you run the filter many times in Q8 to estimate the parameters.

**B.** Implement the smoother. It returns the smoothing distributions $p(z_t|x_{1:T}) = \mathcal{N}(z_t|m_t^s, P_t^s)$.

**C.** The measurements are not equally spaced. Use the time between two consecutive measurements as $\Delta t$, so that $A$ and $Q$ change from one step to the next. Let the filter also skip the update at a step without observation, and say why this is the right thing to do. The record has no missing water level, but such steps let you add hours without data, for example an hourly grid, and get the components there without imputing values. The pressure is not measured at those hours either: interpolate it in time where you need it, for the surge $s_t + \beta \tilde{p}_t$ in Q9 and for the forecast in Q10.

**D.** A missing pressure is not the same thing as a missing water level. Say why, and whether the way you filled the pressure in Q4 is right for the filter.

*Hints for JAX.* Write one step of the filter as a function `step(carry, inputs)` that returns the new $(m_t, P_t)$ and the outputs of the step, and let `jax.lax.scan(step, (m, P), inputs)` loop over the time steps, with `inputs` holding $\Delta t$, the measurement and the pressure of each step. Start the scan from $(m_1^-, P_1^-)$ with $\Delta t = 0$ at the first step, so that $A = I$, $Q = 0$ and the first prediction is the prior $p(z_1)$. The smoother runs backwards with `reverse=True`. For a step without observation, pass a placeholder value with a 0/1 flag, and choose between the updated and the predicted distribution with `jnp.where`. Do not pass a `NaN`: it breaks the gradient, even inside `jnp.where`. Enable double precision with `jax.config.update("jax_enable_x64", True)`, or the log likelihood will be noisy.

</div>

### QUESTION 6.A

In [ ]:
def kalman_filter(theta, omega, H, m1, P1, dts, xs, ps, observed):
    """Kalman filter over a sequence of steps.

    Same for the whole record
        theta = (kappa, sigma_s, sigma_tide, sigma_x, beta)
        omega = angular frequencies of the constituents [rad/h]
        H = observation row
        m1, P1 = mean and covariance of the prior p(z_1)
    One value per step
        dts = time since the previous step [h], 0 at the first step
        xs = water levels, any placeholder value where there is no measurement
        ps = air pressure minus its average [hPa]
        observed = True where there is a measurement

    Returns a dict with the log likelihood "ll" and, for every step,
    "m", "P" (filtering), "m_pred", "P_pred" (one-step-ahead prediction),
    "nu" (innovation) and "S" (its variance)."""
    ...

### QUESTION 6.B

In [ ]:
def rts_smoother(theta, omega, dts, m, P, m_pred, P_pred):
    """Rauch-Tung-Striebel smoother, run backwards on the output of kalman_filter.

    m, P, m_pred, P_pred = the arrays returned by kalman_filter
    Returns the means and covariances of p(z_t | x_{1:T}), for every step."""
    ...

<div class="alert alert-success">

**Q7**. Validate your implementation before touching the real data.

On the real record, the surge is never measured, so you cannot check there whether the filter finds it. On a simulated record, you know the true states, so you can.

Simulate $x_{1:T}$ with your simulator of Q4.D, with the $\theta$ you used there and $z_1$ drawn from your prior. Run the filter and the smoother with that same $\theta$ and the same prior. If the two sides differ, a bad result could come from the difference rather than from your code.

For one record, plot over one month the true surge $s_t$ and its smoothed mean $\mathbb{E}[s_t|x_{1:T}]$ with its 95% band.

Then simulate at least 10 records, each with a different key. Report the empirical coverage of the 95% credible intervals of the surge, the fraction of time steps at which the true surge falls in its interval, for the filter and for the smoother. What value do you expect, and what would a much lower one mean? Compare the average width of the two intervals, and explain the difference.

</div>

<div class="alert alert-success">

**Q8**. Estimate the parameters.

**A.** Write down the log likelihood $\log p(x_{1:T}|\theta)$. Using the decomposition $p(x_{1:T}|\theta) = \prod_{t=1}^{T} p(x_t | x_{1:t-1}, \theta)$ of [Lecture 5](https://glouppe.github.io/dats0001-foundations-of-data-science/?p=lecture5.md#learning-parameters), and the fact that each factor is Gaussian in the linear Gaussian case, show that $$\log p(x_{1:T}|\theta) = -\frac{1}{2}\sum_{t}\left(\log 2\pi S_t + \frac{\nu_t^2}{S_t}\right),$$ where the sum runs over the observed time steps. Explain in one sentence what has happened to the latent states $z_{1:T}$.

**B.** Estimate $\theta$ by maximum a posteriori: maximize this log likelihood plus the log prior of $\log \sigma_x$ from Q4, with the gradient given by `jax`. Fit the positive parameters on a log scale.

</div>

<div class="alert alert-success">

**Q9**. Separate the components.

**A.** Run the smoother with the estimated $\theta$. On a common time axis, plot the measured water level, the smoothed tide (the sum of $\mathbb{E}[c_{j,t}|x_{1:T}]$ over the tidal constituents), the smoothed surge $\mathbb{E}[s_t|x_{1:T}] + \beta \tilde{p}_t$ with its 95% band, and the smoothed slow level (the mean level $\ell$ plus the yearly cycle of Sa and Ssa) with its 95% band. Draw the components at every hour, with the steps without observation of Q6. Show the whole year, and the two days around the highest water level found in Q1.

**B.** Compare your tide plus yearly cycle with `x_pred`, which contains both. Report the RMS difference. Split the difference into a slow part, for example its 30-day running mean, and a fast part, and say where each comes from. Are the differences largest during storms? Compare $\beta$ with the rise of about 1 cm per hPa of pressure drop given in the introduction.

</div>

# Critique

<div class="alert alert-success">

**Q10**. Criticize the model.

**A.** Under the model, the standardized innovations $\nu_t / \sqrt{S_t}$ are independent draws from $\mathcal{N}(0, 1)$. Check this on the data with a histogram, a QQ-plot and their autocorrelation. When does the model fail, and why?

**B.** Simulate a few records from the fitted model at the measurement times, and show them next to the data on the plots of Q1. Start the simulations from the smoothed $p(z_1 | x_{1:T})$: the wide prior of Q4 gives tides of random size. Follow the size $\sqrt{c_{j,t}^2 + e_{j,t}^2}$ of a few constituents along the simulated records, and compare it with its smoothed value on the data. How far does it drift over a year with your $\sigma_\text{tide}$? What does the model reproduce, and what does it miss?

**C.** Forecast the water level. Stop the filter two days before the largest value of `x - x_pred` found in Q1.E, and predict the water level $x_t$ hour by hour over the next 48 hours with the prediction step alone, with its 95% band. Take the measured pressure as if it came from a weather forecast. Compare the forecast with the measurements of those 48 hours, which the filter has not seen. Repeat from at least 20 dates spread through the year, and report how often the measurements fall in the band. Compare the forecast errors with the errors on the surge alone, the smoothed surge $\mathbb{E}[s_t|x_{1:T}]$ minus its forecast: which part of the forecast errors do they explain, and where does the rest come from? Your $\theta$ was fitted on the whole year: say why this makes the forecast look better than it would in practice.

</div>